# Day 29: Linux Internals: Processes & Syscalls

**Week 5 — Cloud SRE**

---

## 📚 Theory: Deep Dive


### 1. The Process Lifecycle
Every process in Linux is created by another process (except `init`/`systemd`).
- **fork()**: Creates a child process by duplicating the calling process. The child gets a new PID.
- **exec()**: Replaces the current process image with a new one (e.g., running `ls`).
- **wait()**: Parent waits for child to finish to collect its exit status.
- **Zombie Processes**: A child that has finished but its parent hasn't called `wait()` yet.

### 2. System Calls (The Interface)
Syscalls are the only way to transition from User Space to Kernel Space.
- **File I/O**: `open`, `read`, `write`, `close`, `lseek`.
- **Process Mgmt**: `fork`, `execve`, `exit`, `getpid`.
- **Memory**: `mmap`, `brk`, `munmap`.
- **Networking**: `socket`, `bind`, `connect`, `send`, `recv`.

### 3. File Descriptors & Standard Streams
- `0`: stdin
- `1`: stdout
- `2`: stderr
- Everything is a file: Pipes, Sockets, and Hardware devices are all accessed via FDs.

### 4. Signals: Kernel-to-Process Communication
- `SIGINT` (2): Interrupt from keyboard (Ctrl+C).
- `SIGTERM` (15): Graceful termination request (Default for `kill`).
- `SIGKILL` (9): Immediate termination (Cannot be caught or ignored).
- `SIGCHLD`: Sent to parent when child dies.



## 🔨 Hands-on Labs


### Lab 1: The Zombie Hunter

Write a Python script that spawns a child process, lets it finish, but the parent sleeps for 10 seconds without calling wait. Observe the zombie state in `ps aux`.


In [ ]:
import os
import time

pid = os.fork()
if pid > 0:
    print(f'Parent {os.getpid()} sleeping... Check "ps aux | grep {pid}" for Z state')
    time.sleep(10)
    os.wait() # Clean up zombie
    print('Parent cleaned up child.')
else:
    print(f'Child {os.getpid()} exiting immediately.')
    os._exit(0)


### Lab 2: Tracing a Mystery Failure

Explain how you would use `strace -e open python script.py` to find exactly which file a Python script is failing to find, even if the error message is vague.


In [ ]:
# Discussion: strace catches the 'ENOENT (No such file or directory)' error from the open() syscall.


## 📝 Day 29 Mastery Checklist

- [ ] Explain the difference between `fork()` and `vfork()`.
- [ ] Identify a process in `D` state (uninterruptible sleep) and explain why it can't be killed.
- [ ] Use `strace` to identify a missing shared library.
- [ ] Calculate the impact of context switching on a high-throughput service.
